<a href="https://colab.research.google.com/github/24dhanush/GUVI_PROJECT/blob/main/final_uber_project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#import data uber_eats
uber_data=pd.read_csv('/content/Uber_Eats_data.csv')

In [ ]:
#Datashape
uber_data.shape


In [ ]:
#DELITE DUPLICATE VALUE
uber_data.drop_duplicates(uber_data)

In [ ]:
#value counts
uber_data['name'].value_counts()

In [ ]:
#ONLY ASSIGNMENT COLUMNS
x=uber_data.drop(columns=['dish_liked', 'votes' ,'phone' ,'rest_type','listed_in(type)','listed_in(city)'])

In [ ]:
#IN  DATA FRAME
uber_final=pd.DataFrame(x)

In [ ]:
uber_final.shape

In [ ]:
uber_final.value_counts()

In [ ]:
uber_final.head(10)

In [ ]:
uber_final.isnull().sum()

REMOVE DUPLICATE DATA FOR ASSIGNMENT FEATURES

In [ ]:
final_ans=uber_final.drop_duplicates(uber_final)

In [ ]:
final_ans.shape

In [ ]:
uber_final.value_counts()

In [ ]:
final_ans['rate'] = final_ans['rate'].astype(str).str.replace('/5', '').str.strip()
print(final_ans['rate'].head())

In [ ]:
final_ans.head()

### order data jeson

In [ ]:
order=pd.read_json('/content/orders.json')

In [ ]:
order.head()

In [ ]:
order.shape

In [ ]:
order.drop_duplicates(order)

In [ ]:
order.value_counts(['restaurant_name','order_value'])

In [ ]:
order.isnull().sum()

update the two dats in sql

In [ ]:
import sqlite3

In [ ]:
connection=sqlite3.connect('uber.db')
cursor=connection.cursor()

In [ ]:
#uber_final table
cursor.execute("""CREATE TABLE IF NOT EXISTS uber_eats(
    name VARCHAR(50),
    online_order VARCHAR(50),
    book_table VARCHAR(50),
    rate FLOAT,
    location VARCHAR(50),
    cuisines VARCHAR(50),
    "approx_cost(for two people)" INT
)""")
connection.commit()
print("Table 'uber_eats' created successfully in SQLite!")

In [ ]:
data_list =final_ans.values.tolist()
query = """
    INSERT INTO uber_eats(name,online_order,book_table,rate,location,cuisines,"approx_cost(for two people)")
    VALUES (?,?,?,?,?,?,?);
"""
cursor.executemany(query, data_list)
cursor.connection.commit()
print("Uber_final data inserted successfully into 'uber_final' table!")

In [ ]:
cursor.execute("""CREATE TABLE IF NOT EXISTS order_list (order_id VARCHAR(50),
restaurant_name VARCHAR(50),
order_date DATE,
order_value INT,
discount_used VARCHAR(50),
payment_method VARCHAR(50)
)""")
connection.commit()
print("Table 'uber' created successfully in SQLite!")

In [ ]:
data_list = order.values.tolist()
query = """
    INSERT INTO order_list (order_id, restaurant_name, order_date,
    order_value, discount_used, payment_method)
    VALUES (?,?,?,?,?,?);
"""
connection.executemany(query, data_list)
connection.commit()
print("Order data inserted successfully using to_list()")

quarys SQL

In [ ]:
query = "SELECT * FROM uber_eats"
cursor.execute(query)
results = cursor.fetchall()
uber_eats_table=pd.DataFrame(results)
print(uber_eats_table)

In [ ]:
query = "SELECT * FROM order_list"
cursor.execute(query)
results = cursor.fetchall()
order_table=pd.DataFrame(results)
print(order_table)

In [ ]:
#uber_eats and order_list

In [ ]:
#1,Which Bangalore locations have the highest average restaurant ratings?
from tabulate import tabulate

query1 = "SELECT location, AVG(rate) AS avg_rating  FROM uber_eats GROUP BY location ORDER BY AVG(rate) DESC LIMIT 1;"
cursor.execute(query1)
result1 = cursor.fetchall()

# Define table headers
headers = ["LOCATION", "RATING"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))


In [ ]:
#2,Which cuisines are most common in Bangalore?
from tabulate import tabulate

query1 = "SELECT cuisines, COUNT(*) AS cuisines_cout FROM uber_eats GROUP BY cuisines ORDER BY cuisines_cout DESC LIMIT 1;"
cursor.execute(query1)
result1 = cursor.fetchall()

headers = ["CUISINES","COUNTS"]

print(tabulate(result1, headers=headers, tablefmt="grid"))

In [ ]:
#3,Which locations are over-saturated with restaurants?
from tabulate import tabulate

query1 = "SELECT location, COUNT(*)name  FROM uber_eats GROUP BY location ORDER BY name DESC  LIMIT 1;"
cursor.execute(query1)
result1 = cursor.fetchall()

headers = [ "LOCATION"," RESTAURANTS"]

print(tabulate(result1, headers=headers, tablefmt="grid"))

In [ ]:
#4,What price range delivers the best customer satisfaction?
from tabulate import tabulate

query2 = """SELECT "approx_cost(for two people)",AVG(rate) AS average_rating FROM uber_eats GROUP BY "approx_cost(for two people)" ORDER BY average_rating DESC LIMIT 1;"""
cursor.execute(query2)
result2 = cursor.fetchall()

headers = ["Approximate Cost (for two people)", "Average Rating"]

print(tabulate(result2, headers=headers, tablefmt="grid"))


In [ ]:
#5,Does online ordering improve restaurant rating?
query2=""" SELECT
               CASE
                  WHEN online_order = 'Yes' THEN 'Online Ordering'
                  ELSE 'No Online Ordering'
               END AS online_order_status,
                AVG(rate) as average_rating
            FROM uber_eats
            GROUP BY online_order_status
            ORDER BY average_rating DESC;"""
cursor.execute(query2)
result1= cursor.fetchall()
print(tabulate (result1, headers=["Online Order Status","Average Rating"], tablefmt="grid"))

In [ ]:
#6,Which location have low rating ?
query2=""" SELECT location , rate FROM uber_eats WHERE rate = (SELECT MIN (rate) FROM uber_eats)ORDER BY rate ASC LIMIT 1 ;"""
cursor.execute(query2)
result=cursor.fetchall()
print(result)

In [ ]:
#7, which restuarent have more order value ?
query= """SELECT restaurant_name, order_value FROM order_list  WHERE order_value = (SELECT MAX (order_value) FROM order_list)ORDER BY order_value ASC LIMIT 1;"""
cursor.execute(query)
result=cursor.fetchall()
print(result)

In [ ]:
#8. Restaurant and Total Revenue ?
query="""SELECT restaurant_name,SUM(order_value) AS total_revenue FROM order_list GROUP BY restaurant_name ORDER BY total_revenue DESC;"""
cursor.execute(query)
result=cursor.fetchall()
print([result])

In [ ]:
#9, which restuarent have more order value?
query= """SELECT  restaurant_name , MAX(order_value) FROM order_list ;"""
cursor.execute(query)
result=cursor.fetchall()
print(result)

In [ ]:
#10. Payment Method Usage?
query= """SELECT payment_method, COUNT(*) AS usage_count FROM order_list GROUP BY payment_method ORDER BY usage_count DESC;"""
cursor.execute(query)
result=cursor.fetchall()
print(result)

In [ ]:
pip install streamlit


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3


# PAGE CONFIG

st.set_page_config(
    page_title="Restaurant Analytics",
    layout="wide"
)


# DATABASE CONNECTION

conn = sqlite3.connect("uber.db")


# FUNCTION TO RUN QUERY

def get_data(query):
    return pd.read_sql_query(query, conn)


# LOAD TABLES

try:
    df1 = pd.read_sql_query("SELECT * FROM uber_eats", conn)
    df2 = pd.read_sql_query("SELECT * FROM order_list", conn)
except:
    df1 = pd.DataFrame()
    df2 = pd.DataFrame()


# SIDEBAR

page = st.sidebar.selectbox(
    "Navigation",
    ["Home", "DataFrames", "SQL Queries"]
)


# HOME PAGE

if page == "Home":

    st.title("🍔 Uber Eats Analytics Dashboard")

    st.subheader("1,Project UBER EATS")
    st.write("Uber Eats Restaurant Analysis")

    st.subheader("About Me")
    st.write("Dhanush DB")

# DATAFRAME PAGE

elif page == "DataFrames":

    st.title("📊 DataFrames")

    option = st.selectbox(
        "Choose DataFrame",
        ["uber_eats", "order_list"]
    )

    if option == "uber_eats":
        st.dataframe(df1)

    elif option == "order_list":
        st.dataframe(df2)

# SQL QUERIES PAGE

elif page == "SQL Queries":

    st.title("📋 SQL Query Results")

    queries={

        "1,Which Bangalore locations have the highest average restaurant ratings?":
        """
        SELECT location,
               AVG(rate) AS avg_rating
        FROM uber_eats
        GROUP BY location
        ORDER BY avg_rating DESC LIMIT 1;
        """,

        "2,Which cuisines are most common in Bangalore?":
        """
        SELECT cuisines,
               COUNT(*) AS cuisine_count
        FROM uber_eats
        GROUP BY cuisines
        ORDER BY cuisine_count DESC LIMIT 1;
        """,

        "3,Which locations are over-saturated with restaurants?":
        """
        SELECT location,
               COUNT(*) AS restaurant_count
        FROM uber_eats
        GROUP BY location
        ORDER BY restaurant_count DESC;
        """,

        "4,What price range delivers the best customer satisfaction?":
        """
        SELECT
            "approx_cost(for two people)" AS price_range,
            AVG(rate) AS average_rating
        FROM uber_eats
        GROUP BY "approx_cost(for two people)"
        ORDER BY average_rating DESC;
        """,

        "5,Does online ordering improve restaurant rating?":
        """
        SELECT
               CASE
                  WHEN online_order = 'Yes' THEN 'Online Ordering'
                  ELSE 'No Online Ordering'
               END AS online_order_status,
               AVG(rate) as average_rating
        FROM uber_eats
        GROUP BY online_order_status
        ORDER BY average_rating DESC;
        """,

        "6,Which location have low rating?":
        """
        SELECT location, rate
        FROM uber_eats
        WHERE rate =
        (
            SELECT MIN(rate)
            FROM uber_eats

        )
        ORDER BY rate ASC
        LIMIT 1;
        """,

        "7, which restuarent have more order value ?":
        """
        SELECT restaurant_name,
               order_value
        FROM order_list
        WHERE order_value =
        (
            SELECT MAX(order_value)
            FROM order_list

        )
        ORDER BY order_value DESC
        LIMIT 1;
        """,

        "8. Restaurant Total Revenue ?":
        """
        SELECT restaurant_name,
               SUM(order_value) AS total_revenue
        FROM order_list
        GROUP BY restaurant_name
        ORDER BY total_revenue DESC;
        """,

        "9. Highest Single Order By Restaurant?":
        """
        SELECT restaurant_name,
               MAX(order_value) AS highest_order
        FROM order_list
        GROUP BY restaurant_name
        ORDER BY highest_order DESC;
        """,

        "10. Payment Method Usage":
        """
        SELECT payment_method,
               COUNT(*) AS usage_count
        FROM order_list
        GROUP BY payment_method
        ORDER BY usage_count DESC;
        """
    }

    selected_query = st.selectbox(
        "Select Query",
        list(queries.keys())
    )

    st.subheader("SQL Query")

    st.code(
        queries[selected_query],
        language="sql"
    )

    result = get_data(
        queries[selected_query]
    )

    st.subheader("Output")

    st.dataframe(result)

# ---------------------------------
# FOOTER
# ---------------------------------
st.sidebar.markdown("---")
st.sidebar.write("Created by Dhanush DB")